# TerraSpectra — Day 1 dataset validation\n\nThis notebook inventories the public wheat Septoria hyperspectral dataset and audits its three annotation CSVs. It does **not** download the large VNIR/SWIR image archives or train a model.\n\nPrimary dataset: [DOI 10.57745/WVP0FJ](https://doi.org/10.57745/WVP0FJ). Read docs/dataset_provenance.md before interpreting fields.\n\nIf using local compute, connect this Colab notebook to your local Jupyter runtime before running the cells.

In [ ]:
from pathlib import Path\nimport os\nimport subprocess\n\nrepo_url = 'https://github.com/rajarathinam274-stack/TerraSpectra-Hyperspectral-Crop-Disease-Forecasting.git'\ncwd = Path.cwd()\nif (cwd / 'scripts' / 'download_metadata.py').exists():\n    repo_dir = cwd\nelse:\n    repo_dir = cwd / 'TerraSpectra-Hyperspectral-Crop-Disease-Forecasting'\n    if not repo_dir.exists():\n        subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)\nos.chdir(repo_dir)\nprint('Repository:', repo_dir)\nsubprocess.run(['git', 'status', '--short', '--branch'], check=True)

In [ ]:
%pip -q install -r requirements.txt

## 1. Download and verify annotation metadata\n\nThe script saves the Dataverse metadata and file inventory, downloads only Data_description.csv, VNIR_data.csv and SWIR_data.csv, verifies advertised file sizes/checksums when available, and records SHA-256 hashes. Large image ZIPs are intentionally skipped.

In [ ]:
!python scripts/download_metadata.py

## 2. Audit annotation schemas\n\nThe audit reports rows, columns, dtypes, missing values, duplicate rows and candidate identity/time/label columns. Column-name matches are hints only; manually confirm their semantics before modelling.

In [ ]:
!python scripts/audit_annotations.py

In [ ]:
!python scripts/inspect_metadata.py

## 3. Run repository tests\n\nA passing configuration test does not prove dataset access. Record actual command output and any API/network error in the Day 1 work log.

In [ ]:
!pytest -q

## 4. Review the outputs\n\nInspect data/metadata/file_inventory.json, download_manifest.json and annotation_audit.json. Before defining a train/test split, identify the biological unit (leaf, plant/pot, image, genotype), time field and infection/treatment labels. Do not claim forecasting unless stable repeated identities and future-state labels are demonstrated.